![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 1. Parte C: Representaciones distribuidas

**Asignatura:** Modelos de Lenguaje
**Bloque:** 2 — Redes Neuronales Recurrentes

---

## Introducción

En la Parte A quedó planteado un problema y no se resolvió. Habías visto que el índice de una palabra en el vocabulario no dice nada sobre su significado: `alarma` y `música` quedaban a seis posiciones de distancia por puro accidente alfabético, mientras que `alarma` y `despertador` quedaban a novecientas. El índice es una etiqueta, no una representación.

La respuesta a ese problema es una idea antigua, de la lingüística de los años cincuenta, y se resume en una frase de J. R. Firth: *"conocerás una palabra por la compañía que mantiene"*. Si dos palabras aparecen rodeadas de las mismas palabras, algo tienen que ver. No hace falta que nadie anote qué significa cada una: alcanza con procesar mucho texto y contar quién aparece cerca de quién.

Este laboratorio convierte esa frase en un vector por palabra, de tres maneras distintas y cada vez con menos ayuda: contando co-ocurrencias a mano, entrenando un modelo (*word2vec* con muestreo negativo), y descargando vectores que otros entrenaron con miles de millones de palabras.

Y en el medio hay un experimento que es el que da sentido a todo lo demás. Vas a entrenar **dos** tablas de vectores sobre el **mismo** corpus de reseñas y con el **mismo** vocabulario, cambiando solo qué le pedimos al modelo que prediga: una aprende a predecir palabras vecinas, la otra a predecir cuántas estrellas tiene la reseña. Las dos son "representaciones distribuidas" y las dos funcionan, pero la geometría que sale es tan distinta que casi no comparten vecinos. Ver esa diferencia es entender de una vez qué es lo que un *embedding* representa: no "el significado" de la palabra, sino exactamente aquello que hizo falta para resolver la tarea con la que se entrenó.

Al completar este laboratorio vas a poder:

- Construir una matriz de co-ocurrencia y medir similitud entre palabras con el coseno.
- Explicar qué captura la razón de probabilidades de co-ocurrencia, que es la idea sobre la que se apoya GloVe.
- Implementar el submuestreo de palabras frecuentes y la extracción de pares centro-contexto.
- Implementar el muestreo negativo y entender por qué convierte un problema de miles de clases en uno binario.
- Escribir y entrenar un modelo *skip-gram* completo en PyTorch.
- Comparar dos geometrías entrenadas sobre el mismo texto y atribuir la diferencia a la señal de entrenamiento.
- Usar vectores preentrenados, resolver analogías, y medir cuánto aportan por transferencia según cuántos datos etiquetados tengas.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **Este laboratorio hace de clase teórica.** Lo único que ya viste es la similitud coseno y la proyección con PCA, de la Clase 6 de la Unidad 1, y el loop de entrenamiento de la Unidad 1. Todo lo demás —el submuestreo, el muestreo negativo, el *skip-gram*— se explica acá, en el enunciado de cada ejercicio, antes de pedirte que lo implementes. Leelos con calma: el enunciado no es solo la consigna, es el material.
- **Este laboratorio corre entero en CPU**, en unos cinco minutos. El entrenamiento más largo son tres épocas de unos treinta segundos cada una. Si tenés GPU disponible podés usarla, pero no hace falta.
- Las celdas de setup dejan listos el tokenizador y la clase `Vocabulario` con los que se codifica el corpus. Usalos sin modificarlos: si los reemplazás por otra implementación, los números no van a coincidir con los esperados.
- **Fijá las semillas que te pide cada enunciado.** El submuestreo y el muestreo negativo son aleatorios, y sin semilla nada de lo que imprimas va a ser comparable con nada.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las tres celdas que siguen ya vienen resueltas. No hay nada que completar en ellas, pero **hay que ejecutarlas** en orden antes de empezar, y conviene leer las dos últimas porque definen los nombres que usan todos los ejercicios.

La primera importa las librerías. La segunda descarga el corpus de reseñas. La tercera arma el vocabulario y fija la dimensión de los vectores.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
import os
import math
import time
import urllib.request
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

print(f"Versión de PyTorch: {torch.__version__}")

### El corpus: reseñas de Amazon en español

Son 200.000 reseñas de productos etiquetadas con la cantidad de estrellas que puso quien las escribió, de 1 a 5 y perfectamente balanceadas. Cada reseña trae el título y el cuerpo separados por un salto de línea doble.

Es el **mismo** corpus que usa el notebook de teoría de la Unidad 1, y no es casualidad: en el Ejercicio 6 vas a comparar los vectores que entrenes acá contra la tabla de *embeddings* del clasificador de estrellas. Para que esa comparación signifique algo, el texto y el vocabulario tienen que ser exactamente los mismos.

La celda descarga los tres *splits* del Hub de Hugging Face —22 MB, unos segundos— y los deja en `train`, `val` y `test`. Después imprime un resumen de largos y una reseña de ejemplo.

In [ ]:
# ─── Setup: el corpus de reseñas de Amazon en español ───────────────────────
HUB = ("https://huggingface.co/datasets/mteb/AmazonReviewsClassification"
       "/resolve/main/es")


def leer_split(nombre):
    """Lee un split del corpus desde el Hub de Hugging Face."""
    return pd.read_parquet(f"{HUB}/{nombre}-00000-of-00001.parquet")


train = leer_split("train")
val   = leer_split("validation")
test  = leer_split("test")

print(f"entrenamiento: {len(train):>7,} reseñas")
print(f"validación:    {len(val):>7,} reseñas")
print(f"prueba:        {len(test):>7,} reseñas")
print(f"estrellas:     {sorted(int(e) + 1 for e in train.label.unique())}  (label + 1)")
print()

largos = train.text.str.split().str.len()
print(f"palabras por reseña: media {largos.mean():.1f}, mediana {largos.median():.0f}, "
      f"percentil 95 {np.percentile(largos, 95):.0f}, máximo {largos.max()}")
print()
print("una reseña de 1 estrella:")
print(" ", repr(train.text.iloc[0]))

### El tokenizador, el vocabulario y la dimensión

Esta celda deja disponibles el tokenizador `tok_simple` y la clase `Vocabulario`, y con ellos construye `vocab` sobre el corpus de entrenamiento. Vienen provistos para que los números de este laboratorio sean comparables entre todos. Deja además dos constantes que vas a ver en todos los ejercicios: `V`, el tamaño del vocabulario, y `DIM`, la dimensión de los vectores.

Hay dos decisiones tomadas acá que conviene que entiendas antes de seguir:

- **`freq_min=10`**, mucho más alto que el `freq_min=2` de la Parte A. Para aprender un vector de 100 dimensiones que signifique algo, una palabra tiene que aparecer muchas veces. Con diez apariciones ya es discutible; con dos es imposible, y esas filas terminarían siendo ruido con el que después vamos a medir similitudes.
- **`DIM = 100` está acá y no en el ejercicio donde se usa**, porque tiene que ser la misma para el *skip-gram* y para el clasificador supervisado: si no, la comparación del Ejercicio 6 no vale.

In [ ]:
# ─── Setup: el tokenizador, el vocabulario y la dimensión ────────────────────
# El tokenizador y la clase Vocabulario vienen en un módulo auxiliar de la
# materia, que bajamos acá.
URL_PIPELINE = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/utils-v1/lab1a_pipeline.py")

if not os.path.exists("lab1a_pipeline.py"):
    urllib.request.urlretrieve(URL_PIPELINE, "lab1a_pipeline.py")

from lab1a_pipeline import tok_simple, Vocabulario


vocab = Vocabulario(train.text, tokenizador=tok_simple, freq_min=10)
V = len(vocab)

# La dimensión de los vectores que vas a entrenar. Está acá y no en el
# ejercicio donde se usa porque tiene que ser la MISMA para el skip-gram y para
# el clasificador supervisado: si no, la comparación del Ejercicio 6 no vale.
DIM = 100

en_vocab = sum(f for p, f in vocab.contador.items() if p in vocab.stoi)
totales = sum(vocab.contador.values())

print(vocab)
print(f"formas distintas en el corpus: {len(vocab.contador):,}")
print(f"formas en el vocabulario:      {V:,}  (incluye <pad> y <unk>)")
print(f"dimensión de los embeddings:   {DIM}")
print(f"tokens totales:                {totales:,}")
print(f"cobertura de apariciones:      {en_vocab / totales:.2%}")
print()
print("las 15 más frecuentes:", vocab.itos[2:17])

---
## Sección A: La hipótesis distribucional, sin entrenar nada

Antes de entrenar nada conviene ver que la idea funciona con nada más que contar. Un vector por palabra se puede construir contando: la fila de la matriz de co-ocurrencia de una palabra **ya es** una representación distribuida, y el coseno entre dos filas ya mide algo parecido a la similitud.

Este único ejercicio hace las dos cosas que valen la pena de ese enfoque directo: mostrar que funciona, y mostrar dónde falla. Ese límite es el que van a atacar los métodos que vienen después.

### Ejercicio 1 — Contar la compañía: co-ocurrencia y razón de probabilidades

**Objetivo:** Construir vectores de palabra contando co-ocurrencias, medir similitud con el coseno, y calcular la razón de probabilidades sobre la que se apoya GloVe.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — la matriz de co-ocurrencia sobre un corpus juguete.** Diez oraciones alcanzan para ver el mecanismo entero.

```python
juguete = [
    "el teléfono tiene una pantalla excelente",
    "el teléfono tiene una pantalla pésima",
    "la tablet tiene una pantalla excelente",
    "la tablet tiene una pantalla pésima",
    "compré el teléfono y la batería dura poco",
    "compré la tablet y la batería dura poco",
    "el libro tiene una historia excelente",
    "el libro tiene una historia pésima",
    "devolví el teléfono porque la batería dura poco",
    "devolví el libro porque la historia es pésima",
]
```

1. Armá el vocabulario del corpus juguete —ordenado alfabéticamente, sin tokens especiales— en una lista llamada `vocab_j`, con su diccionario inverso `idx_j` y su tamaño en `Vj`. Atención: `V` a secas ya está tomada por la celda de preparación, donde es el tamaño del vocabulario del corpus real, así que la del juguete necesita otro nombre.

2. Construí la matriz de co-ocurrencia `C`, de forma `(Vj, Vj)`, donde `C[i, j]` es la cantidad de veces que la palabra `j` aparece a **2 posiciones o menos** de la palabra `i` (ventana de 2 a cada lado, sin contarse a sí misma).

3. Normalizá cada fila de `C` para que tenga norma 1, guardá el resultado en `Cn`, y calculá en `S` la matriz de similitudes: el producto de `Cn` por su transpuesta. Con las filas normalizadas ese producto **es** el coseno entre cada par de palabras. Escribí después la función `vecinos_j(palabra, k=4)`, que devuelva las `k` palabras de coseno más alto con su coseno, sin contar la palabra misma.

4. Imprimí los vecinos de `excelente`, `teléfono`, `tablet` y `batería`, y después el coseno de estos cuatro pares: `(excelente, pésima)`, `(teléfono, tablet)`, `(teléfono, libro)` y `(excelente, batería)`. **Observá el primero con atención antes de seguir.**

En la salida vas a ver dos resultados que sorprenden, y los dos son correctos. Uno es el coseno de `excelente` con `pésima`, que es sobre el que pregunta el análisis. El otro es que el vecino más cercano de `teléfono` termina siendo `libro`, y no `tablet`: con diez oraciones lo que domina es la plantilla en la que cada palabra aparece —`el ___ tiene una`, `devolví el ___ porque`—, no de qué habla. Es la misma causa vista en otro par.

> **Pista:** Dos bucles anidados sobre cada oración tokenizada alcanzan. La ventana se recorta con `max(0, i - 2)` y `min(len(s), i + 3)`.

In [ ]:
# Tu código aquí

**Parte B — la razón de probabilidades, sobre el corpus real.** La fila de co-ocurrencia dice con quién aparece una palabra. Comparar **dos** filas dividiéndolas dice algo más fino: qué distingue a una palabra de la otra.

Sobre esa observación se apoya **GloVe** (*Global Vectors*), uno de los métodos clásicos de representación distribuida junto con *word2vec*. Su punto de partida es que lo informativo de una matriz de co-ocurrencia no son los conteos sino los **cocientes**: para dos palabras `a` y `b` y una palabra de sondeo `w`, la razón entre la probabilidad de que `w` aparezca cerca de `a` y la de que aparezca cerca de `b`. GloVe entrena vectores para que su producto punto reproduzca el logaritmo de esos conteos. Acá no vamos a entrenarlo: vamos a calcular esa razón a mano y ver qué muestra.

5. Codificá el corpus de entrenamiento, una sola vez, en una lista de listas llamada `corpus`: una lista por reseña, con los índices de `vocab` de sus tokens. Descartá los tokens que no estén en el vocabulario en lugar de mandarlos a `<unk>`. **Esta estructura la usa todo el resto del laboratorio**, así que conviene dejarla bien.

6. Escribí la función `contextos_de(palabra, ventana=5)`, que recorra `corpus` y devuelva un `Counter` con los **índices** que aparecen a `ventana` posiciones o menos de `palabra`, sin contar la posición de `palabra` misma.

7. Para el par `a = "bueno"`, `b = "malo"`, imprimí una tabla con una fila por cada palabra de sondeo de la lista de abajo, y cuatro columnas: la palabra, `p(w|a)`, `p(w|b)` y la razón `p(w|a) / p(w|b)`. Ordenala por la razón, de mayor a menor.

```python
sondas = ["precio", "calidad", "producto", "pero", "genial",
          "recomiendo", "devolver", "vale", "dinero", "malísimo"]
```

8. **Repetí la tabla para el par `a = "caro"`, `b = "barato"`** y compará las dos: en una, las razones se despliegan en varios órdenes de magnitud; en la otra, se agrupan alrededor de 1.

> **Pista 1:** `p(w|a)` es la cantidad de veces que `w` aparece en el contexto de `a`, dividida por el total de tokens de contexto de `a`.

> **Pista 2:** Como la tabla se pide dos veces, conviene escribir una función que reciba `a` y `b` y la imprima.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) En el corpus juguete, el coseno entre `excelente` y `pésima` da 0,945: dos antónimos quedan casi en el mismo punto. Explicá por qué el método los junta, y por qué eso **no** es un error de implementación sino una propiedad de lo que estás midiendo.

b) Sobre el corpus real, la razón `p(w|a)/p(w|b)` se despliega en varios órdenes de magnitud para `bueno`/`malo` y se agrupa cerca de 1 para `caro`/`barato`. ¿Qué información da la razón que no daban las dos probabilidades por separado, y qué te dice la diferencia entre los dos pares?

*(Escribí tu respuesta acá)*

---
## Sección B: Entrenar los vectores (*word2vec* con muestreo negativo)

Contar funciona, pero la matriz de co-ocurrencia de un corpus real tiene `V × V` celdas —para nuestro vocabulario, más de 240 millones— casi todas en cero, y hay que guardarla entera antes de poder usarla.

*Word2vec* invierte el problema. En lugar de construir la matriz y después factorizarla, entrena directamente los vectores con descenso por gradiente sobre una tarea de predicción: dada una palabra central, predecir cuáles son sus vecinas. Los vectores no son el resultado que le pedimos al modelo, son el **subproducto** de resolver esa tarea, y ahí está la clave del método.

Los cuatro ejercicios de esta sección construyen el método completo, en el orden en que hay que pensarlo: preparar el flujo de datos, inventar la señal negativa, escribir el modelo y entrenarlo.

### Ejercicio 2 — Preparar el flujo: submuestreo y pares centro-contexto

**Objetivo:** Implementar el submuestreo de palabras frecuentes y la extracción de pares, que es lo que convierte un corpus de texto en ejemplos de entrenamiento.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — el submuestreo.** El problema de partida es que `de` aparece 222.174 veces y `tallaje` 351. Sin corregir nada, el modelo pasaría la mayor parte del tiempo aprendiendo que `de` aparece cerca de `la`, que es cierto y no le sirve a nadie. La receta del artículo original descarta cada aparición de una palabra con probabilidad

$$P_{\text{descartar}}(w) = \max\left(0,\; 1 - \sqrt{\frac{t}{f(w)}}\right)$$

donde $f(w)$ es la frecuencia relativa de la palabra en el corpus y $t$ es un umbral, típicamente $10^{-4}$. Notá que la decisión se toma **por aparición**, no por palabra: `de` no desaparece del corpus, aparece muchas menos veces.

1. Contá cuántas veces aparece cada índice del vocabulario en `corpus` y guardá esas frecuencias **absolutas** en un array llamado `frec`, de tamaño `V`. Dividiéndolo por su suma obtenés las frecuencias relativas $f(w)$, y con ellas calculá `p_descartar`, un array del mismo tamaño con la probabilidad de descarte de cada índice. Guardá `frec`: el Ejercicio 3 lo vuelve a usar.

2. Aplicá el submuestreo con `rng = np.random.default_rng(0)` y construí `corpus_sub`, descartando las reseñas que queden con menos de 2 tokens. Imprimí cuántos tokens había, cuántos quedaron y qué porcentaje sobrevive.

3. Imprimí, para las 8 palabras más frecuentes y para `batería`, `excelente` y `tallaje`, su frecuencia absoluta y su probabilidad de descarte.

> **Pista 1:** `np.add.at(frec, c, 1)` acumula los conteos de una lista de índices sin bucle de Python.

> **Pista 2:** Para el punto 2, sortear un vector de uniformes con `rng.random(len(a))` y comparar contra `p_descartar[a]` resuelve la reseña entera de una vez.

In [ ]:
# Tu código aquí

**Parte B — los pares.** Cada aparición de cada palabra genera un ejemplo de entrenamiento por cada vecina dentro de su ventana.

4. Construí dos arrays de `numpy`, `centros` y `contextos`, recorriendo `corpus_sub`. Para cada posición, sorteá un **tamaño de ventana entre 1 y 5 inclusive** con el mismo `rng`, y emití un par por cada vecina dentro de esa ventana.

5. Imprimí cuántos pares salieron y el promedio de pares por token.

6. **Verificá que funciona:** decodificá a palabras la reseña `corpus_sub[0]` e imprimí, también en texto, los **primeros 10 pares** `(centro, contexto)` de `centros`/`contextos`. Como el recorrido empieza por esa reseña, esos diez pares son los suyos, y podés leer contra la reseña impresa si la ventana quedó bien puesta.

> **Pista:** El tamaño de ventana aleatorio no es un capricho de implementación: es lo que hace que las palabras más cercanas al centro pesen más, porque entran en más de los sorteos.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) El submuestreo descarta el 63% de los tokens del corpus. Explicá por qué eso **mejora** los vectores en vez de empeorarlos, y qué le pasaría al entrenamiento si no lo aplicaras.

b) La fórmula descarta apariciones, no palabras. Explicá qué se rompería si en cambio sacaras del vocabulario a las palabras más frecuentes.

*(Escribí tu respuesta acá)*

### Ejercicio 3 — Muestreo negativo: de 15.548 clases a una decisión binaria

**Objetivo:** Entender y construir el mecanismo que hace que *word2vec* sea entrenable, y armar los lotes que va a consumir el modelo.

**Enunciado:**

El *skip-gram* "puro" predice, dada una palabra central, cuál de las **15.548** palabras del vocabulario es la vecina. Eso es un softmax sobre 15.548 clases en cada uno de los 11 millones de pares: el denominador solo ya vuelve el entrenamiento inviable.

El muestreo negativo cambia la pregunta. En vez de *"¿cuál de todas las palabras es la vecina?"*, pregunta *"¿este par de palabras es un par real del corpus, o lo inventé yo?"*. Eso es una clasificación **binaria**, con una sigmoide en lugar de un softmax, y por cada par real alcanza con inventar unos pocos falsos.

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — la distribución de la que se sortean los falsos.** No se sortean uniformemente ni con la frecuencia real, sino con la frecuencia elevada a 3/4 y renormalizada:

$$P_{\text{neg}}(w) = \frac{f(w)^{3/4}}{\sum_{w'} f(w')^{3/4}}$$

1. Construí el tensor `pesos_neg` de tamaño `V` con esa distribución, a partir de las frecuencias absolutas `frec` del ejercicio anterior, poniendo en cero las posiciones de `<pad>` y `<unk>` antes de renormalizar.

2. Imprimí una tabla comparativa con una fila por cada palabra de la lista `["de", "que", "muy", "precio", "batería", "excelente", "pésimo", "tallaje"]` y cuatro columnas: la frecuencia, `P(w)` (la frecuencia relativa), `P_neg(w)`, y la razón entre las dos últimas. Esa última columna es la que muestra qué hace el exponente.

In [ ]:
# Tu código aquí

**Parte B — los lotes.** Cada ejemplo de entrenamiento va a ser un par real más `K = 5` inventados.

3. Escribí la función `armar_lote(idx)` que, dado un tensor de índices sobre `centros`/`contextos`, devuelva tres tensores: `c` de forma `(B,)` con los centros; `ctx` de forma `(B, 1+K)` con el contexto real en la **columna 0** y `K` negativos sorteados de `pesos_neg` en el resto; y `y` de forma `(B, 1+K)` con las etiquetas, que valen 1 en la columna 0 y 0 en las demás.

4. Fijá `torch.manual_seed(0)`, armá con `armar_lote` el lote de los **cuatro primeros pares** (los índices 0 a 3) y mostralo **en texto**: para cada fila, la palabra central, la de contexto real y las cinco negativas. Dejá los tres tensores en `c`, `ctx` e `y`, que el Ejercicio 4 los va a reusar.

> **Pista 1:** `torch.multinomial(pesos_neg, B * K, replacement=True).view(B, K)` sortea todos los negativos del lote de una vez.

> **Pista 2:** Las etiquetas son siempre las mismas para todos los lotes del mismo tamaño, así que conviene construirlas una vez afuera y reusarlas.

> **Pista 3:** Un negativo sorteado puede resultar ser una vecina real por casualidad. Con un vocabulario de 15.548 palabras eso ocurre muy pocas veces, y el método lo tolera; no hace falta verificarlo.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Observá la última columna de la tabla de `P_neg`. Explicá qué le hace el exponente 3/4 a la distribución, y por qué sortear los negativos con la frecuencia real `P(w)` daría peores vectores.

b) El *skip-gram* con softmax resuelve un problema de 15.548 clases; con muestreo negativo, 6 clasificaciones binarias por par. Explicá de dónde sale el ahorro de cómputo, y qué se pierde a cambio.

*(Escribí tu respuesta acá)*

### Ejercicio 4 — El modelo: dos tablas y un producto punto

**Objetivo:** Implementar el *skip-gram*, que es el modelo más pequeño que vas a escribir en todo el curso: no tiene ninguna capa lineal.

**Enunciado:**

El modelo entero son **dos tablas de embeddings** y un producto punto. La puntuación que le da al par (centro $c$, contexto $o$) es

$$s(c, o) = \mathbf{v}_c \cdot \mathbf{u}_o$$

donde $\mathbf{v}$ sale de la tabla de centros y $\mathbf{u}$ de la tabla de contextos. Esa puntuación es un *logit*: alto si el modelo cree que el par es real.

1. Implementá la clase `SkipGram`, subclase de `nn.Module`, cuyo constructor `__init__(self, n_vocab, dim=DIM)` cree dos atributos, `self.centro` y `self.contexto`, las dos capas `nn.Embedding` de `n_vocab × dim`. Inicializá los pesos de `self.centro` con una distribución uniforme en $[-0{,}5/\text{dim},\; 0{,}5/\text{dim}]$ y los de `self.contexto` en **ceros**. (`DIM` vale 100 y viene de la celda de preparación: usalo, porque el Ejercicio 6 necesita que las dos tablas tengan la misma dimensión.)

2. Implementá el método `forward(self, c, ctx)`: recibe `c` de forma `(B,)` y `ctx` de forma `(B, 1+K)`, y devuelve los *logits* de forma `(B, 1+K)`. Hacelo con `torch.bmm`, sin bucles.

3. Fijá `torch.manual_seed(0)`, instanciá la clase en una variable llamada `modelo` e imprimí la cantidad de parámetros de cada tabla y el total.

4. **Verificación de cordura.** Pasá por el modelo sin entrenar el lote de 4 ejemplos del ejercicio anterior (`c` y `ctx`) y verificá que los *logits* dan **todos exactamente cero**. Después calculá la pérdida contra `y` con `nn.BCEWithLogitsLoss()` y comprobá que da $\ln 2 = 0{,}6931$. Imprimí las dos cosas.

> **Pista 1:** Fijar a mano la inicialización de una capa no apareció en la Unidad 1, así que va explicado. Se hace con las funciones del módulo `nn.init`, que modifican un tensor de parámetros en el lugar: `nn.init.uniform_(t, a, b)` llena `t` con una uniforme entre `a` y `b`, y `nn.init.zeros_(t)` lo pone en cero. El tensor de parámetros de una `nn.Embedding` es su atributo `.weight`.

> **Pista 2:** `torch.bmm` (*batch matrix multiply*) tampoco apareció en la Unidad 1. Multiplica lotes de matrices: toma `(B, n, m)` y `(B, m, p)` y devuelve `(B, n, p)`, haciendo un producto de matrices independiente por cada elemento del lote. Acá lo que querés es, para cada ejemplo, multiplicar los `1+K` vectores de contexto —que forman una matriz `(1+K, E)`— por el único vector central, que hay que llevar a la forma `(E, 1)` con un `unsqueeze`. El resultado sale `(B, 1+K, 1)` y un `squeeze` lo deja en la forma que pide el enunciado.

> **Pista 3:** Para el punto 4, pensá cuánto vale un producto punto contra un vector de ceros, y qué probabilidad le asigna la sigmoide a un *logit* de cero.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) El modelo le da a cada palabra **dos** vectores, uno de centro y uno de contexto. Explicá qué problema aparecería si usaras una sola tabla para los dos papeles.

b) La verificación da exactamente $\ln 2$ por cómo inicializaste `self.contexto`. Explicá la cadena que lleva de esa inicialización al número, y por qué esta verificación detecta bugs que el entrenamiento por sí solo no detectaría.

*(Escribí tu respuesta acá)*

### Ejercicio 5 — Entrenar, y examinar qué aprendió

**Objetivo:** Entrenar el *skip-gram* completo y evaluar los vectores por lo único que se puede evaluar sin más datos: si sus vecinos tienen sentido.

**Enunciado:**

1. **Escribí el loop de entrenamiento.** Es el mismo loop de la Unidad 1, con tres cambios: tres épocas, lotes de 4.096 pares y `Adam` con `lr=5e-3`. La pérdida, en cambio, es otra: como cada salida es una decisión binaria —¿este par es real o inventado?— no va entropía cruzada multiclase sino `nn.BCEWithLogitsLoss`, que aplica la sigmoide internamente igual que `CrossEntropyLoss` aplicaba el softmax. En cada época barajá los pares con `torch.randperm` y armá los lotes con `armar_lote`. Fijá `torch.manual_seed(0)` antes de crear el modelo. Imprimí la pérdida media y el tiempo de cada época.

   > Son unos 2.700 lotes por época y unos 30 segundos por época en CPU. Si querés ver el progreso adentro de la época, imprimí cada 500 lotes.

2. **Guardá en `E_dist` la tabla de *embeddings* de los centros** del modelo entrenado, desprendida del grafo de cómputo y con cada fila normalizada a norma 1. Con las filas normalizadas, el coseno entre dos palabras es el producto punto de sus filas, y el coseno de una palabra contra todo el vocabulario es un solo producto matriz-vector.

3. **Escribí la función `vecinos(palabra, k=8, tope=6000, tabla=None)`**, que devuelva las `k` palabras más parecidas a `palabra` por similitud coseno. El parámetro `tope` restringe la búsqueda a las `tope` palabras más frecuentes, para que no aparezcan palabras poco frecuentes con vectores mal entrenados; excluí también `<pad>`, `<unk>` y la palabra misma. El parámetro `tabla` es el que permite consultar **otra** tabla de vectores con la misma función: cuando vale `None` se usa `E_dist`, y el Ejercicio 6 la va a llamar con una segunda tabla, así que conviene dejarlo previsto desde acá.

4. **Imprimí los vecinos** de: `excelente`, `pésimo`, `caro`, `batería`, `talla`, `libro`, `envío`, `devolver` y `cargador`.

5. **Volvé sobre el Ejercicio 1.** Imprimí el coseno entre `caro` y `barato` en la tabla entrenada, y comparalo con lo que habías visto en la tabla de razones de probabilidad.

> **Pista:** Esta función es la misma que la Clase 6 escribe en *6.2 Palabras vecinas* sobre la tabla del clasificador de estrellas; lo único que cambia es de qué tabla salen los vectores. Revisá también cómo evita el bucle sobre el vocabulario, que es lo que hace el punto 2.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Elegí **dos** de las palabras que consultaste y explicá qué tipo de relación agrupó el modelo en cada caso. No todas agrupan lo mismo: hay sinónimos, hay variantes morfológicas, hay palabras del mismo tema que no son intercambiables.

b) El vecino más cercano de `caro` es `barato`. Conectá este resultado con la tabla de razones del Ejercicio 1 y con el coseno de 0,945 del corpus juguete: los tres son el mismo fenómeno. ¿Qué consecuencia práctica tiene para usar estos vectores en análisis de sentimiento?

*(Escribí tu respuesta acá)*

---
## Sección C: Qué representa realmente un *embedding*

Ya tenés vectores entrenados y sus vecinos tienen sentido. La tentación es concluir que el modelo "aprendió el significado de las palabras". Esta sección es para desarmar esa conclusión.

El Ejercicio 6 entrena una segunda tabla sobre el **mismo texto** y el **mismo vocabulario**, cambiando una sola cosa: qué le pedimos que prediga. Si un *embedding* representara el significado, las dos tablas deberían parecerse. No se parecen en nada, y ese resultado es el que hay que entender.

Después, los ejercicios 7 y 8 salen del corpus: vectores entrenados con miles de millones de palabras, qué se puede hacer con ellos, y cuánto sirven de verdad cuando los usás en una tarea propia.

### Provisto: la otra tabla de embeddings

La celda que sigue no hay que completarla: entrena sobre este corpus el clasificador de la Parte B, con la misma arquitectura —tabla de *embeddings*, promedio enmascarado, una capa oculta y una de salida— y dos ajustes que impone la tarea: la capa de salida tiene 5 clases en vez de 18, porque acá se predicen estrellas, y la tabla usa `DIM = 100`, la misma dimensión que el *skip-gram*. El texto y el `vocab` son los mismos que venís usando.

Además de la clase `ClasificadorEstrellas`, la celda deja definidos los nombres que reaparecen en el Ejercicio 8: `L`, el largo al que se recortan las reseñas; los tensores `X_ent`, `y_ent`, `X_test` e `y_test`; la función `accuracy(m, X, y)`, que evalúa un modelo por lotes; y `E_sup`, la tabla de *embeddings* entrenada y normalizada fila por fila para poder medir cosenos.

Todo lo que cambia entre las dos tablas es la señal con la que se entrenó:

| tabla | qué aprende a predecir | de dónde sale la señal |
|---|---|---|
| `E_dist` (*skip-gram*) | qué palabras aparecen cerca | el texto solo |
| `E_sup` (clasificador) | cuántas estrellas tiene la reseña | las etiquetas |

Ese es el experimento del Ejercicio 6, así que corré esta celda antes de llegar ahí. Tarda unos 15 segundos.

In [ ]:
# ─── Provisto: la OTRA tabla de embeddings ──────────────────────────────────
import torch.nn.functional as F

L = 48                                   # el percentil 95 del corpus es 76


class ClasificadorEstrellas(nn.Module):
    """El modelo de la Parte B, con 5 clases en vez de 18."""

    def __init__(self, n_vocab, dim_emb=DIM, dim_oculta=128, n_clases=5, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(n_vocab, dim_emb, padding_idx=pad_id)
        self.oculta = nn.Linear(dim_emb, dim_oculta)
        self.salida = nn.Linear(dim_oculta, n_clases)

    def promediar(self, x):
        vectores = self.embedding(x)
        mascara = (x != self.pad_id).unsqueeze(-1).float()
        return (vectores * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)

    def forward(self, x):
        return self.salida(F.relu(self.oculta(self.promediar(x))))


def accuracy(m, X, y, lote=2048):
    """Proporción de aciertos de `m` sobre (X, y), evaluando por lotes."""
    m.eval()
    with torch.no_grad():
        return torch.cat([(m(X[i:i + lote]).argmax(1) == y[i:i + lote]).float()
                          for i in range(0, len(X), lote)]).mean().item()


X_ent = vocab.codificar_lote(train.text, L)
y_ent = torch.tensor(train.label.values)
X_test = vocab.codificar_lote(test.text, L)
y_test = torch.tensor(test.label.values)

torch.manual_seed(0)
sup = ClasificadorEstrellas(V, pad_id=vocab.pad_id)
opt_s = torch.optim.Adam(sup.parameters(), lr=1e-3)
crit_s = nn.CrossEntropyLoss()

t0 = time.time()
for epoca in range(4):
    perm = torch.randperm(len(X_ent))
    for i in range(0, len(X_ent), 256):
        idx = perm[i:i + 256]
        perdida = crit_s(sup(X_ent[idx]), y_ent[idx])
        opt_s.zero_grad()
        perdida.backward()
        opt_s.step()

acc = accuracy(sup, X_test, y_test)

E_sup = sup.embedding.weight.detach()
E_sup = E_sup / E_sup.norm(dim=1, keepdim=True).clamp(min=1e-8)

print(f"clasificador de estrellas entrenado en {time.time() - t0:.0f} s")
print(f"accuracy en prueba: {acc:.1%}   (azar con 5 clases: 20,0%)")
print(f"tabla de embeddings: {tuple(sup.embedding.weight.shape)}, "
      f"la misma forma que la del skip-gram")

### Ejercicio 6 — Dos geometrías sobre el mismo texto

**Objetivo:** Comparar las dos tablas y atribuir la diferencia a lo único que las distingue, que es la señal de entrenamiento.

**Enunciado:**

Tenés `E_dist` (tu *skip-gram*) y `E_sup` (la tabla del clasificador de estrellas), las dos normalizadas, las dos de `15.548 × 100`, entrenadas sobre el mismo texto con el mismo vocabulario. Cambia una sola cosa: qué predice cada modelo.

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — los vecinos, lado a lado.**

1. Imprimí una comparación con una fila por palabra y dos columnas —los 4 vecinos más cercanos en cada tabla— para: `excelente`, `pésimo`, `caro`, `batería`, `talla`, `libro`, `cargador` y `devolver`.

2. **Cuantificá el desacuerdo.** Para esas mismas palabras, calculá cuántos de sus 10 vecinos más cercanos comparten las dos tablas, y promediá ese conteo sobre las ocho palabras. Imprimí el promedio, que va de 0 a 10.

3. **Observá los cosenos de pares elegidos.** Imprimí una tabla con `cos` en las dos geometrías para: `(bueno, malo)`, `(caro, barato)`, `(bien, mal)`, `(excelente, pésimo)` y `(excelente, perfecto)`.

In [ ]:
# Tu código aquí

**Parte B — las dos nubes.**

4. Proyectá las dos tablas a dos dimensiones con PCA y graficalas **lado a lado**, en una figura de dos paneles, sobre esta lista de palabras:

```python
PALABRAS = ["excelente", "perfecto", "genial", "estupendo", "recomendable", "encantada",
            "pésimo", "horrible", "malísimo", "defectuoso", "decepcionante", "estafa",
            "caro", "barato", "precio", "calidad",
            "batería", "pantalla", "cargador", "talla", "camiseta", "libro",
            "cocina", "sonido", "envío", "tamaño"]
```

5. **Coloreá cada punto por la cantidad promedio de estrellas** de las reseñas donde aparece esa palabra, con el mapa `RdYlGn`. Anotá cada punto con su palabra, y poné título, grilla y barra de color en los dos paneles.

> **Pista 1:** Los dos puntos son los de *6.3 La geometría del espacio aprendido*, de la Clase 6: ahí está el PCA con `torch.pca_lowrank` —hay que centrar la submatriz antes de proyectar— y el coloreado por estrellas. Lo único nuevo acá es hacerlo dos veces y comparar.

> **Pista 2:** Para el color te alcanza con una pasada sobre `train` acumulando, por palabra, la suma de estrellas y en cuántas reseñas aparece. Contá una vez por reseña, no una vez por aparición.

> **Pista 3:** Fijá `vmin=2.0, vmax=4.0` en el `scatter` para que la escala de color sea la misma en los dos paneles. Sin eso, la comparación visual no vale.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Las dos tablas no comparten **ni un solo** vecino en el top 10, y `caro`/`barato` pasa de coseno 0,676 a −0,070. Explicá qué organiza cada geometría, y por qué la señal de entrenamiento —y no la arquitectura ni el corpus— es la que produce la diferencia.

b) En el panel supervisado, palabras como `batería`, `talla` o `cargador` quedan agrupadas en el centro y sus vecinos no tienen sentido, mientras que en el distribucional tienen vecinos coherentes. Explicá por qué, y qué conclusión general sacás sobre qué representa un *embedding*.

*(Escribí tu respuesta acá)*

### Provisto: los vectores preentrenados

La celda que sigue no hay que completarla: descarga vectores de **fastText** para el español, entrenados sobre Common Crawl y Wikipedia. Son del orden de $10^{11}$ palabras de entrenamiento, contra los 6 millones de nuestro corpus: cinco órdenes de magnitud más.

El archivo original pesa 1,3 GB y trae 2.000.000 de tokens, así que usamos un recorte: los 150.000 más frecuentes en minúscula, guardados en `float16`. Son 81 MB y se descargan en unos segundos; la precisión que se pierde al usar `float16` no afecta nada de lo que vamos a medir.

Al terminar vas a tener `pre_itos` y `pre_stoi` —el mapeo entre token e índice de esta tabla, que **no** es el mismo que el de `vocab`— y la matriz `E_pre`, ya normalizada fila por fila.

In [ ]:
# ─── Provisto: descarga de los vectores preentrenados ───────────────────────
URL_VECTORES = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/vectores-v1/vectores_es_150k.npz")
ARCHIVO = "vectores_es_150k.npz"

if not os.path.exists(ARCHIVO):
    print("bajando los vectores preentrenados...")
    urllib.request.urlretrieve(URL_VECTORES, ARCHIVO)

datos = np.load(ARCHIVO, allow_pickle=True)
pre_itos = list(datos["tokens"])
pre_stoi = {t: i for i, t in enumerate(pre_itos)}

# Los pasamos a float32 y los normalizamos una vez: todo lo que sigue es coseno.
E_pre = torch.tensor(datos["vectors"].astype(np.float32))
E_pre = E_pre / E_pre.norm(dim=1, keepdim=True).clamp(min=1e-8)

print(f"{len(pre_itos):,} tokens, dimensión {E_pre.shape[1]}")
print(f"los 10 más frecuentes: {pre_itos[:10]}")

### Ejercicio 7 — Vectores preentrenados: vecinos, analogías y agujeros

**Objetivo:** Usar vectores entrenados con cinco órdenes de magnitud más de texto, y medir qué se gana y qué falta.

**Enunciado:**

1. **Escribí la función `vecinos_pre(palabra, k=8)`**, el equivalente de `vecinos` pero sobre `E_pre` y `pre_stoi`. Imprimí los vecinos de `rey`, `excelente`, `pésimo`, `batería` y `cargador`, y compará con los que te había dado tu propio modelo; no hace falta que escribas la comparación.

2. **Escribí la función `analogia(a, b, c, k=3)`**, que resuelva *"a es a b como c es a ?"* buscando las palabras más cercanas por coseno al vector $\mathbf{b} - \mathbf{a} + \mathbf{c}$. Excluí del resultado a las tres palabras de la consulta.

3. **Probala** con: `("hombre", "rey", "mujer")`, `("madrid", "españa", "parís")`, `("bueno", "mejor", "malo")` y `("comer", "comí", "beber")`. Imprimí las tres mejores respuestas de cada una con su coseno.

4. **Medí la cobertura.** ¿Qué proporción de las **formas** de tu vocabulario está en el archivo preentrenado, y qué proporción de las **apariciones** del corpus queda cubierta? Imprimí las dos cifras, cuántas formas faltan, y las 15 formas faltantes más frecuentes.

> **Pista 1:** Normalizá el vector de la analogía antes de multiplicar por `E_pre`, o los cosenos no van a estar en la escala correcta.

> **Pista 2:** Para excluir palabras del resultado, poné su similitud en −1 antes del `topk`, igual que hiciste con el `tope` en el Ejercicio 5.

> **Pista 3:** La cobertura de apariciones se pondera con `vocab.contador`: no es lo mismo que falte una palabra que aparece 3 veces que una que aparece 3.000.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Las analogías funcionan buscando el vecino de $\mathbf{b} - \mathbf{a} + \mathbf{c}$. Explicá qué propiedad tiene que tener el espacio para que eso dé algo sensato, y por qué tu propio modelo del Ejercicio 5 casi seguro no la tiene.

b) La cobertura de apariciones es del 99%, pero falta el 4% de las formas, y casi todas las que faltan son números. Explicá por qué la cobertura de apariciones es la cifra que importa, y qué haría un modelo basado en **subpalabras** con una palabra que no está en la tabla.

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Transferencia: ¿cuánto sirven de verdad?

**Objetivo:** Medir qué aportan los vectores preentrenados como inicialización de una tarea propia, y descubrir que la respuesta depende de cuántos datos etiquetados tengas.

**Enunciado:**

Vas a volver a la clase `ClasificadorEstrellas` que quedó definida en la celda de preparación, ahora con `dim_emb=300` para que entren los vectores de fastText, y a comparar tres maneras de inicializar su tabla de *embeddings*.

1. **Construí los tensores de validación** `X_val` e `y_val`, a partir de `val`, igual que la celda de preparación construyó los de entrenamiento y prueba y con el mismo `L`.

2. **Construí la matriz de inicialización** `M_pre` de forma `(V, 300)`: para cada forma de `vocab.itos`, su vector preentrenado si existe; para las que no existen, un vector al azar `normal(0, sigma)` con `sigma` igual al desvío estándar de `E_pre`. La fila de `<pad>` va en ceros. Usá `np.random.default_rng(0)`.

3. **Barajá `X_ent`/`y_ent` antes de recortar**, con `torch.randperm` y semilla 0, y guardá el resultado en `X_baraja`/`y_baraja`. El corpus viene ordenado por estrellas: quedarte con las primeras 2.000 filas sin barajar te daría una sola clase.

4. **Escribí la función `correr(nombre, pesos, congelar, n, epocas)`**, que entrene el clasificador sobre las primeras `n` reseñas de `X_baraja` y devuelva la accuracy de prueba **en la época de mejor accuracy de validación**. La época se elige mirando validación y nunca prueba: elegirla por prueba contamina la estimación con la partición que después se reporta, y para eso justamente hay una partición de validación. Tenés `accuracy(m, X, y)` lista desde la celda de preparación. Las tres configuraciones son:

   - `pesos=None` → tabla al azar, entrenable.
   - `pesos=M_pre, congelar=True` → preentrenada, `requires_grad = False`.
   - `pesos=M_pre, congelar=False` → preentrenada, ajuste fino.

   Fijá `torch.manual_seed(0)` al empezar cada corrida, y pasale al optimizador solo los parámetros con `requires_grad`.

5. **Corré las tres configuraciones con `n = 2.000` (20 épocas) y con `n = 200.000` (4 épocas)**, e imprimí una tabla con las seis accuracies. Agregá una columna con la cantidad de parámetros entrenables de cada configuración.

> **Pista 1:** Cargar una matriz de pesos en una `nn.Embedding` ya creada y congelarla no apareció en la Unidad 1, así que va explicado: los pesos se copian sobre `m.embedding.weight.data`, y para que dejen de recibir gradiente se pone `m.embedding.weight.requires_grad = False`. Las dos cosas se hacen sobre el mismo atributo `.weight`.

> **Pista 2:** Con la tabla congelada solo se entrenan las dos capas lineales, unos 39.000 parámetros contra 4,7 millones. Esa diferencia es parte de lo que explica el resultado.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) La ventaja de inicializar con vectores preentrenados es de 4,7 puntos con 2.000 reseñas y de 1,3 con 200.000. Explicá por qué se achica, y qué regla general de la transferencia se sigue de eso.

b) La tabla congelada **cambia de lado**: con 2.000 reseñas le gana a la inicialización al azar (44,2% contra 41,9%) y con 200.000 le pierde (52,0% contra 54,7%). Explicá el cruce usando la cantidad de parámetros entrenables de cada configuración, y qué le falta a los vectores de fastText para esta tarea en particular.

*(Escribí tu respuesta acá)*

### Provisto: tres límites de estos vectores

La celda que sigue no hay que completarla: es el experimento que da pie al último ejercicio. Muestra tres cosas que estos vectores no pueden hacer —una palabra polisémica tiene un solo vector, las asociaciones del texto de entrenamiento quedan grabadas en la geometría, y el orden de las palabras sigue sin existir— y para cada una imprime la evidencia.

Usa las funciones `vecinos_pre` y `analogia` que escribiste en el Ejercicio 7, así que si todavía no lo resolviste va a dar `NameError`. Observá la salida con atención antes de responder.

In [ ]:
# ─── Provisto: tres límites, en tres líneas ─────────────────────────────────
# 1. Una palabra, un vector, aunque la palabra tenga dos sentidos.
print("polisemia")
for p in ["muñeca", "sierra", "banco", "gato"]:
    print(f"  {p:9s} -> {', '.join(vecinos_pre(p, 7))}")

# 2. Lo que estaba en el texto de entrenamiento, queda en los vectores.
#    Atención con el segundo caso: la respuesta morfológicamente correcta existe en
#    la tabla, y aun así el modelo prefiere otra.
print("\nasociaciones aprendidas del corpus")
for a, b, c in [("hombre", "ingeniero", "mujer"),
                ("hombre", "jefe", "mujer"),
                ("él", "médico", "ella")]:
    print(f"  {a} : {b} :: {c} : ?  ->  {analogia(a, b, c, k=4)}")
print(f"  para comparar, dónde quedó 'médica': puesto "
      f"{[w for w, _ in analogia('él', 'médico', 'ella', k=30)].index('médica') + 1}")

# 3. El orden sigue sin existir.
print("\ndos frases, mismas palabras")
f1, f2 = "el envío fue rápido pero el producto es malo", \
         "el envío fue malo pero el producto es rápido"
for f in (f1, f2):
    ids = [pre_stoi[t] for t in tok_simple(f) if t in pre_stoi]
    print(f"  {f!r}\n     promedio de sus vectores: norma {E_pre[ids].mean(0).norm():.4f}")
iguales = torch.allclose(
    E_pre[[pre_stoi[t] for t in tok_simple(f1) if t in pre_stoi]].mean(0),
    E_pre[[pre_stoi[t] for t in tok_simple(f2) if t in pre_stoi]].mean(0))
print(f"  ¿los dos promedios son idénticos? {iguales}")

### Ejercicio 9 — Tres límites de una tabla de vectores

**Objetivo:** Cerrar el laboratorio identificando qué es lo que esta representación no puede hacer, que es lo que motiva todo lo que sigue en la materia.

**Enunciado:**

La celda de arriba muestra tres límites de asignarle **un vector fijo a cada palabra**. Respondé, apoyándote en la salida:

1. **Polisemia.** Las cuatro palabras de la primera lista tienen dos sentidos en español, pero la salida muestra **dos comportamientos distintos**: en dos de ellas los vecinos vienen de los dos sentidos mezclados, y en las otras dos uno de los sentidos directamente no aparece. Identificá cuáles son cuáles, explicá qué le pasa al vector en cada caso, y por qué ninguna cantidad de datos de entrenamiento lo arregla mientras la tabla tenga una sola fila por palabra.

2. **Asociaciones del corpus.** De las tres analogías, la primera devuelve la respuesta esperable y las otras dos no. En el caso de `médico`, notá que la forma femenina correcta **está** en la tabla y aun así queda por debajo. Explicá de dónde sale esa asociación y por qué llamarla "sesgo del modelo" describe mal el problema. Nombrá una consecuencia concreta de usar estos vectores sin más en un sistema que tome decisiones sobre personas.

El tercer límite no hay que analizarlo, porque ya lo viste en la Parte B: las dos frases del final tienen las mismas palabras y significados opuestos, y su promedio de vectores es idéntico. La información no se pierde en los vectores sino **después**, en la suma que los combina, y como la suma es conmutativa da lo mismo qué tan buenos sean los vectores de entrada. Lo que tiene que cambiar es la operación que los junta, y de eso se ocupa la unidad que sigue.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Fijé todas las semillas que pedían los enunciados, así que mis números son reproducibles.
- [ ] La verificación del Ejercicio 4 da `ln(2) = 0,6931` y los *logits* iniciales son cero.
- [ ] Los vecinos del Ejercicio 5 tienen sentido: si me salieron palabras sin relación, algo está mal en la extracción de pares o en el modelo.
- [ ] La figura del Ejercicio 6 tiene los dos paneles con la **misma escala de color**, título, grilla y barra de color.
- [ ] La tabla del Ejercicio 8 tiene las seis accuracies y todas están bastante por encima del 20% del azar.
- [ ] Respondí las ocho preguntas de análisis (Ej. 1 a 8) y el análisis final del Ejercicio 9.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Construiste representaciones distribuidas de tres maneras, con cada vez menos ayuda:

- **Contando**: la matriz de co-ocurrencia, el coseno entre filas, y la razón de probabilidades sobre la que se apoya GloVe.
- **Entrenando**: el *skip-gram* completo, con submuestreo, muestreo negativo y una pérdida binaria, sobre once millones de pares.
- **Descargando**: vectores de fastText entrenados con cinco órdenes de magnitud más de texto, analogías, y transferencia a una tarea propia.

Y en el medio quedó el resultado que importa. Las dos tablas del Ejercicio 6 se entrenaron sobre el mismo texto, con el mismo vocabulario y la misma dimensión, y no comparten ni un vecino de cada diez. Un *embedding* no representa "el significado" de una palabra: representa **lo que hizo falta para resolver la tarea con la que se entrenó**. La distribucional sabe de qué se habla y confunde `caro` con `barato`; la supervisada los separa y no sabe qué es un cargador.

Todo esto sigue teniendo el mismo techo que la Parte B, y el Ejercicio 9 lo dejó a la vista: una palabra, un vector, y un promedio que borra el orden. *"El envío fue rápido pero el producto es malo"* y *"el envío fue malo pero el producto es rápido"* siguen siendo la misma entrada.

Lo que viene son las dos maneras de romper ese techo: procesar la secuencia arrastrando un estado, y dejar que cada palabra mire a las demás.